# Rollout-training ablation data

Load all available rollout-training-ablation JSONL results into one tidy DataFrame. This notebook intentionally performs no analysis or visualization.

The DataFrame retains records from incomplete in-progress files; downstream analysis can decide whether to filter to completed configurations.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / 'derived').is_dir() else Path.cwd().parent
RESULTS_DIR = REPO_ROOT / 'derived' / 'results' / 'rollout_training_ablation'
FILENAME_PATTERN = re.compile(
    r'^rollouttrain_(?P<dynamic>diffusion|wave|coupled_oscillator)_'
    r'(?P<topology>ring|grid|torus|doorway|swisscheese)_'
    r'models(?P<models>\d+)_epochs(?P<epochs>\d+)_nodes(?P<num_nodes>\d+)_'
    r'episodes(?P<num_episodes>\d+)_bins(?P<num_bins>\d+)_events(?P<events_per_bin>\d+)_'
    r'dropint(?P<raindrop_interval>\d+)_tau(?P<event_threshold>[0-9p]+)_'
    r'trainroll(?P<rollout_train_steps>\d+)_rollhorizon(?P<rollout_horizon>\d+)_'
    r'dt(?P<synthetic_dt>[0-9p]+)_gamma(?P<synthetic_gamma>[0-9p]+)_'
    r'omega(?P<synthetic_omega>[0-9p]+)_forcescale(?P<synthetic_force_scale>[0-9p]+)_'
    r'seed(?P<seed>\d+)\.jsonl$'
)


In [ ]:
def decode_numeric_tag(value: str) -> float:
    return float(value.replace('p', '.'))


def load_rollout_training_results(results_dir: Path) -> pd.DataFrame:
    paths = sorted(results_dir.glob('rollouttrain_*.jsonl'))
    if not paths:
        raise FileNotFoundError(f'No rollout-training JSONLs found in {results_dir}')

    records: list[dict] = []
    for path in paths:
        match = FILENAME_PATTERN.fullmatch(path.name)
        if match is None:
            raise ValueError(f'Unexpected rollout-training filename: {path.name}')

        raw_metadata = match.groupdict()
        metadata = {
            key: (decode_numeric_tag(value) if key.startswith(('event_threshold', 'synthetic_')) else value)
            for key, value in raw_metadata.items()
        }
        metadata.update({
            key: int(value)
            for key, value in raw_metadata.items()
            if key not in {'dynamic', 'topology', 'event_threshold', 'synthetic_dt', 'synthetic_gamma', 'synthetic_omega', 'synthetic_force_scale'}
        })

        with path.open(encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, start=1):
                row = json.loads(line)
                row['source_file'] = path.name
                row['source_line'] = line_number
                row.update(metadata)
                records.append(row)

    # Flatten scalar metric dictionaries one level while retaining detailed
    # rollout curves as object-valued columns.
    return pd.json_normalize(records, sep='.', max_level=1)


rollouttrain_df = load_rollout_training_results(RESULTS_DIR)
rollouttrain_df
